# BPDA-PGD robustness sweep for Tucker purification

This Colab runs the repository's `imagenet_bpda_attack.py` rather than copying its attack implementation into the notebook. It attacks the full Tucker-defended classifier with `advertorch.BPDAWrapper`: the forward pass performs the real randomized Tucker reconstruction, while the backward pass uses the identity surrogate.

The final figure reports **BPDA robust accuracy** (accuracy of Tucker-purified adversarial images) against the Tucker masking/noise percentage.

## 1. Colab setup

In Colab select **Runtime → Change runtime type → T4 GPU**. This notebook assumes the cloned project already contains the complete `data/imagenet val (small)` directory, including its `val` class folders and label metadata.

In [ ]:
!nvidia-smi
!pip -q install tensorly-torch timm pycocotools tqdm
# advertorch provides LinfPGDAttack and BPDAWrapper. --no-deps keeps Colab's CUDA-enabled PyTorch.
!pip -q install --no-deps git+https://github.com/BorealisAI/advertorch.git

from pathlib import Path

REPO_URL = 'https://github.com/RomanGaraev/Image-Restoration.git'
PROJECT_DIR = Path('/content/Image-Restoration')
if not PROJECT_DIR.exists():
    !git clone --depth 1 {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}

# Keep the notebook thin: the BPDA implementation lives in the cloned project.
attack_script = PROJECT_DIR / 'imagenet_bpda_attack.py'
if not attack_script.is_file():
    raise FileNotFoundError(
        f'{attack_script} is absent from this GitHub revision. Commit and push '
        'imagenet_bpda_attack.py (and this notebook) to the repository, then rerun this cell.'
    )


## 2. Configure the sweep

The dataset is read directly from `data/imagenet val (small)`. Start with the smoke-test values, then increase `MAX_BATCHES`, `PGD_STEPS`, and `EOT_SAMPLES` for final measurements.

For fair comparisons, keep the same evaluated image subset and all attack parameters fixed across the noise sweep.

In [ ]:
# Tucker defense sweep. Values are fractions; the graph labels them as percentages.
NOISE_PERCENTS = [0.30, 0.40, 0.50, 0.60]
RANK = 70
DECOMPOSE_STEPS = 30

# Pixel-space, untargeted Linf BPDA-PGD parameters.
EPS = 8 / 255
ALPHA = 2 / 255
PGD_STEPS = 10       # Smoke test; use 40 for the final attack.
EOT_SAMPLES = 1      # Use 2-4 for a stronger attack on the randomized defense.
BATCH_SIZE = 2       # Keep small: Tucker runs inside each PGD step.
MAX_BATCHES = 2      # Smoke test; use a fixed larger value (or -1) for final results.

assert NOISE_PERCENTS and all(0 <= rho <= 1 for rho in NOISE_PERCENTS)
dataset_root = PROJECT_DIR / 'data' / 'imagenet val (small)'
assert (dataset_root / 'val').is_dir(), f'Missing ImageNet validation images: {dataset_root / "val"}'

import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime in Colab, then rerun.'
torch.backends.cudnn.benchmark = True
print(f'PyTorch: {torch.__version__}; GPU: {torch.cuda.get_device_name(0)}')
print(f'ImageNet validation directory: {dataset_root / "val"}')


## 3. Run the repository BPDA experiment

This is a single CLI invocation so the attack remains maintained in `imagenet_bpda_attack.py`. It writes machine-readable results that the next cell plots. The reported `adversarial_defended_accuracy` is the required adaptive BPDA metric.

In [ ]:
import json
import subprocess
import sys

RESULTS_FILE = PROJECT_DIR / 'bpda_tucker_sweep.json'
command = [
    sys.executable, str(attack_script),
    '--dataset-name', 'imagenet',
    '--model-id', 'resnet50',
    '--device', 'cuda',
    '--method', 'tucker',
    '--noise-percent', *map(str, NOISE_PERCENTS),
    '--rank', str(RANK),
    '--decompose-steps', str(DECOMPOSE_STEPS),
    '--batch-size', str(BATCH_SIZE),
    '--eps', str(EPS),
    '--alpha', str(ALPHA),
    '--pgd-steps', str(PGD_STEPS),
    '--eot-samples', str(EOT_SAMPLES),
    '--max-batches', str(MAX_BATCHES),
    '--log-file', str(RESULTS_FILE),
]
print('Running:', ' '.join(command))
subprocess.run(command, check=True, cwd=PROJECT_DIR)

with RESULTS_FILE.open(encoding='utf-8') as handle:
    experiment = json.load(handle)

observed_rhos = {float(value) for value in experiment['results']}
assert observed_rhos == set(NOISE_PERCENTS), (
    f'Expected {NOISE_PERCENTS}, but script returned {sorted(observed_rhos)}. '
    'Check that the repository revision includes the noise-sweep fix.'
)
experiment['attack']


## 4. BPDA robust accuracy vs. noise percentage

The plotted accuracy is measured after the adversarial image is purified. `clean_defended_accuracy` in the table is included to expose the clean-accuracy cost of each Tucker setting.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

rows = [
    {'noise_percent': 100 * float(rho), **metrics}
    for rho, metrics in experiment['results'].items()
]
results = pd.DataFrame(rows).sort_values('noise_percent').reset_index(drop=True)
display(results[[
    'noise_percent', 'clean_defended_accuracy', 'adversarial_defended_accuracy', 'mean_linf'
]].rename(columns={
    'noise_percent': 'noise (%)',
    'clean_defended_accuracy': 'clean Tucker accuracy (%)',
    'adversarial_defended_accuracy': 'BPDA robust accuracy (%)',
    'mean_linf': 'mean Linf',
}))

fig, axis = plt.subplots(figsize=(7, 4.5))
axis.plot(
    results['noise_percent'],
    results['adversarial_defended_accuracy'],
    marker='o',
    linewidth=2,
    color='#1565c0',
)
axis.set_xlabel('Tucker masking / noise (%)')
axis.set_ylabel('Accuracy on BPDA adversarial examples (%)')
axis.set_title('Adaptive BPDA robustness of Tucker purification')
axis.set_xticks(results['noise_percent'])
axis.set_ylim(0, 100)
axis.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

results.to_csv(PROJECT_DIR / 'bpda_tucker_sweep.csv', index=False)


## Reading the result

A high curve is meaningful only if the attack is sufficiently strong. Before reporting a result, rerun the same fixed subset with at least 40 PGD steps and increase EOT when the randomized Tucker masking causes unstable values. If robust accuracy decreases as either setting is strengthened, use the stronger setting.